# 01 — Basismodelle auf SisFall

Entspricht Kapitel 5, Abschnitt *Modellentwicklung*.

Drei aufeinander aufbauende Ansaetze auf identischem, subjekt-
unabhaengigem Testsatz:

| | Verfahren | Art |
|---|---|---|
| 1 | Schwellwert (Bourke) | kein ML, feste Regel — Referenzpunkt |
| 2 | Random Forest | klassisches ML, 18 Merkmale, interpretierbar |
| 3 | 1D-CNN | Deep Learning auf dem Rohsignal |

Legt am Ende `fall_model.keras` ab; alle weiteren Notebooks bauen
darauf auf.

> **Wichtig:** Schwellwert und Random Forest arbeiten auf den
> **rohen** Fenstern in g und °/s. Die Z-Normalisierung entfernt
> gerade die absoluten Amplituden, auf denen ein Schwellwert beruht.
> Nur das CNN erhaelt normalisierte Eingaben.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q scipy scikit-learn tensorflow openpyxl

import os, sys, shutil, time, json
from pathlib import Path
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Gemeinsame Bibliothek holen (einzige Quelle fuer alle Loader)
# fall_lib laden: zuerst GitHub, sonst Kopie auf Drive. Schlaegt beides
# fehl, wird abgebrochen -- ohne fall_lib waeren alle Ergebnisse falsch
# (andere Fensterlaenge, anderes Heruntertasten).
import urllib.request, shutil as _sh
_LIB_URL = "https://raw.githubusercontent.com/Alyelb/ML-Modell-for-Fall-detection/main/fall_lib.py"
_LIB_DRIVE = "/content/drive/MyDrive/ML_Fall_Detection/fall_lib.py"
try:
    urllib.request.urlretrieve(_LIB_URL, "/content/fall_lib.py")
    print("[fall_lib] von GitHub geladen")
except Exception as _e:
    print(f"[fall_lib] GitHub nicht erreichbar ({_e}) -- nutze Drive-Kopie")
    _sh.copy(_LIB_DRIVE, "/content/fall_lib.py")
sys.path.insert(0, "/content")
import fall_lib as fl
import importlib; importlib.reload(fl)
# Pflichtfunktionen pruefen
for _f in ("new_run", "build_windows", "znorm", "WIN_LEN"):
    assert hasattr(fl, _f), f"fall_lib unvollstaendig ({_f} fehlt) -- Datei pruefen."
print("[fall_lib] Fensterlaenge", fl.WIN_LEN, "bei", fl.FS_TARGET, "Hz")

BASE  = "/content/drive/MyDrive/ML_Fall_Detection"
fl.RESULTS_ROOT = f"{BASE}/results"
MODEL = f"{BASE}/results/fall_model.keras"      # einheitlicher Modellpfad

print("Konfiguration:", fl.default_config())

Mounted at /content/drive
Konfiguration: {'fs_target': 50, 'window_sec': 2.0, 'overlap': 0.5, 'window_len': 100, 'step': 50, 'kfall_fall_tasks': 'T20-T34'}


In [2]:
def hole(name, ziel):
    """Kopiert einen Datensatz von Drive auf die lokale Platte.
    Drive ist fuer tausende Kleindateien zu langsam."""
    src = f"{BASE}/{name}"
    if Path(ziel).exists():
        print(f"[ok] {name} bereits lokal"); return ziel
    if not Path(src).exists():
        print(f"[!] {src} nicht gefunden. Vorhanden:")
        for p in sorted(Path(BASE).iterdir()):
            print("   ", p.name)
        return None
    t = time.time(); shutil.copytree(src, ziel)
    print(f"[ok] {name} kopiert in {time.time()-t:.0f}s")
    return ziel

## Daten laden

In [3]:
SIS = hole("SisFall_dataset", "/content/SisFall")
trials = fl.load_sisfall(SIS)

# X_raw bleibt unnormalisiert (fuer Schwellwert und Random Forest)
X_raw, y, groups, acts = fl.build_windows(trials, mode="standard")
print(f"\nFenster: {X_raw.shape}, Stuerze: {int(y.sum())}")

run = fl.new_run("01_basismodelle")
tr, te = fl.subject_split(groups, y, test_size=0.25)
print(f"Train {len(tr)} ({len(np.unique(groups[tr]))} Prob.) | "
      f"Test {len(te)} ({len(np.unique(groups[te]))} Prob.)")
ergebnisse = {}

[ok] SisFall_dataset kopiert in 517s
[SisFall] 4504 Trials (1797 Sturz, 2707 ADL), 38 Probanden

Fenster: (51405, 100, 6), Stuerze: 1797
[Lauf] /content/drive/MyDrive/ML_Fall_Detection/results/01_basismodelle_2026-08-26_0838
Train 38974 (28 Prob.) | Test 12431 (10 Prob.)


## 1 — Schwellwertverfahren

Ein Fenster gilt als Sturz, wenn der Betrag der Beschleunigung einen
festen Schwellwert ueberschreitet. Der Schwellwert wird auf den
Trainingsdaten bestimmt (Youden-Index) und anschliessend unveraendert
auf den Testsatz angewendet.

In [4]:
best_thr, sweep = fl.threshold_sweep(X_raw[tr], y[tr])

print("\nAnwendung auf den Testsatz:")
pred = fl.threshold_predict(X_raw[te], acc_threshold=best_thr)
ergebnisse["threshold"] = fl.evaluate_predictions(
    y[te], pred, f"Schwellwert ({best_thr:.1f} g)")
ergebnisse["threshold"]["threshold_g"] = best_thr
ergebnisse["threshold_sweep"] = sweep

   Schwelle      Se      Sp     Acc
  ----------------------------------
        1.4   1.000   0.518   0.537
        1.5   1.000   0.575   0.591
        1.6   0.999   0.629   0.643
        1.7   0.999   0.673   0.685
        1.8   0.997   0.704   0.716
        1.9   0.994   0.732   0.742
        2.0   0.992   0.753   0.762
        2.1   0.987   0.770   0.778
        2.2   0.980   0.787   0.794
        2.3   0.973   0.799   0.806
        2.4   0.969   0.811   0.817
        2.5   0.964   0.821   0.827
        2.6   0.958   0.829   0.834
        2.7   0.950   0.839   0.843
        2.8   0.941   0.850   0.853
        2.9   0.926   0.860   0.863
        3.0   0.913   0.871   0.872
        3.1   0.901   0.882   0.883
        3.2   0.887   0.893   0.893
        3.3   0.875   0.903   0.902
        3.4   0.860   0.913   0.911
        3.5   0.844   0.922   0.919
        3.6   0.818   0.931   0.927
        3.7   0.807   0.938   0.933
        3.8   0.790   0.945   0.939
        3.9   0.767   0.952

## 2 — Random Forest

18 Merkmale, je neun aus Beschleunigung und Drehrate. Alle beruhen
auf dem Vektorbetrag und sind damit **rotationsinvariant**: Die
Einbaulage des Sensors in der Sohle beeinflusst sie nicht — fuer eine
Einlegesohle wesentlich, da die Ausrichtung nicht garantiert werden
kann.

In [5]:
F = fl.extract_features(X_raw)
print(f"Merkmalsmatrix: {F.shape}\n")

rf = fl.train_random_forest(F[tr], y[tr], n_estimators=200)
ergebnisse["random_forest"] = fl.evaluate_predictions(
    y[te], rf.predict(F[te]), "Random Forest")

Merkmalsmatrix: (51405, 18)

  Random Forest               Se=0.927  Sp=0.996  Acc=0.994  FP=   48  FN=  22


### Merkmalswichtigkeit

Der Random Forest ist im Gegensatz zum CNN interpretierbar. Die
Verteilung der Wichtigkeit auf Beschleunigungs- und
Gyroskopmerkmale liefert einen empirischen Beitrag zur
Sensorempfehlung aus Kapitel 2.

In [6]:
wichtig = fl.feature_importance(rf, top=10)
ergebnisse["feature_importance"] = wichtig

gyr_anteil = sum(r["importance"] for r in wichtig
                 if r["sensor"] == "Gyroskop")
print(f"\n  Gesamtanteil der Gyroskopmerkmale: {gyr_anteil:.1%}")
ergebnisse["gyro_importance_share"] = round(float(gyr_anteil), 4)

  Rang Merkmal           Wichtigkeit Sensor          
  ----------------------------------------------------
     1 gyr_std               0.23619 Gyroskop        
     2 acc_skew              0.12874 Beschleunigung  
     3 gyr_range             0.12049 Gyroskop        
     4 gyr_max               0.10414 Gyroskop        
     5 acc_max               0.06605 Beschleunigung  
     6 acc_kurtosis          0.06178 Beschleunigung  
     7 gyr_energy            0.05865 Gyroskop        
     8 gyr_rms               0.04847 Gyroskop        
     9 acc_range             0.04460 Beschleunigung  
    10 acc_std               0.03603 Beschleunigung  

  Unter den fuenf wichtigsten Merkmalen sind 3 gyroskopbasiert.

  Gesamtanteil der Gyroskopmerkmale: 62.3%


### Kontrolle der Rotationsinvarianz

Die Merkmale werden nach einer zufaelligen dreidimensionalen Drehung
erneut berechnet. Bleiben sie unveraendert, ist die Invarianz
bestaetigt.

In [7]:
from scipy.spatial.transform import Rotation

R = Rotation.random(random_state=0).as_matrix().astype(np.float32)
X_rot = X_raw.copy()
X_rot[:, :, 0:3] = X_raw[:, :, 0:3] @ R.T
X_rot[:, :, 3:6] = X_raw[:, :, 3:6] @ R.T

F_rot = fl.extract_features(X_rot)
gleich = (rf.predict(F[te]) == rf.predict(F_rot[te])).mean()
print(f"  Groesste absolute Merkmalsabweichung: "
      f"{np.abs(F - F_rot).max():.2e}")
print(f"  Unveraenderte Vorhersagen: {gleich:.2%}")
ergebnisse["rotation_invariance"] = {
    "unchanged_predictions": round(float(gleich), 4)}

  Groesste absolute Merkmalsabweichung: 2.00e+00
  Unveraenderte Vorhersagen: 100.00%


## 3 — 1D-CNN

Lernt die Merkmale selbst aus dem Rohsignal (100 × 6). Hier werden
z-normalisierte Eingaben verwendet.

In [8]:
X = fl.znorm(X_raw)

model = fl.build_cnn(n_channels=6)
model.summary()
fl.train(model, X[tr], y[tr], epochs=30, verbose=1)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 96, 32)         │           992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 96, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 48, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 44, 64)         │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 44, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 22, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 18, 128)        │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 18, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,601 (240.63 KB)

 Trainable params: 61,153 (238.88 KB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/30
518/518 ━━━━━━━━━━━━━━━━━━━━ 14s 15ms/step - accuracy: 0.9589 - loss: 0.0982 - val_accuracy: 0.9537 - val_loss: 0.1091 - learning_rate: 0.0010
Epoch 2/30
518/518 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9864 - loss: 0.0403 - val_accuracy: 0.8928 - val_loss: 0.3899 - learning_rate: 0.0010
Epoch 3/30
518/518 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9901 - loss: 0.0271 - val_accuracy: 0.9877 - val_loss: 0.0370 - learning_rate: 0.0010
Epoch 4/30
518/518 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9941 - loss: 0.0155 - val_accuracy: 0.9858 - val_loss: 0.0349 - learning_rate: 0.0010
Epoch 5/30
518/518 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9912 - loss: 0.0244 - val_accuracy: 0.9788 - val_loss: 0.0597 - learning_rate: 0.0010
Epoch 6/30
518/518 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9917 - loss: 0.0221 - val_accuracy: 0.9938 - val_loss: 0.0162 - learning_rate: 0.0010
Epoch 7/30
518/518 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9967 - loss: 0.0086 

<Sequential name=sequential, built=True>

In [9]:
print("\nSisFall — subjektunabhaengiger Testsatz\n")
ergebnisse["cnn"] = fl.evaluate(model, X[te], y[te], "1D-CNN")
ergebnisse["cnn"]["params"] = int(model.count_params())

model.save("/content/fall_model.keras")
shutil.copy("/content/fall_model.keras", MODEL)
shutil.copy("/content/fall_model.keras", run / "fall_model.keras")
print(f"\nModell abgelegt unter {MODEL}")


SisFall — subjektunabhaengiger Testsatz

  1D-CNN                      Se=0.987  Sp=0.998  Acc=0.998  FP=   21  FN=   4

Modell abgelegt unter /content/drive/MyDrive/ML_Fall_Detection/results/fall_model.keras


## Vergleich der drei Verfahren

In [10]:
print(f"{'Verfahren':<24} {'Se':>8} {'Sp':>8} {'Acc':>8} "
      f"{'FP':>7} {'FN':>6}")
print("-"*66)
for key, name in [("threshold", "Schwellwert (Bourke)"),
                  ("random_forest", "Random Forest"),
                  ("cnn", "1D-CNN")]:
    r = ergebnisse[key]
    print(f"{name:<24} {r['Se']:>8.3f} {r['Sp']:>8.3f} "
          f"{r['Acc']:>8.3f} {r['FP']:>7d} {r['FN']:>6d}")

Verfahren                      Se       Sp      Acc      FP     FN
------------------------------------------------------------------
Schwellwert (Bourke)        0.967    0.829    0.832    2074     10
Random Forest               0.927    0.996    0.994      48     22
1D-CNN                      0.987    0.998    0.998      21      4


## int8-TFLite-Export

Fuer die Portierung auf den ESP32-C6.

In [11]:
kb = fl.export_tflite_int8(model, X[tr],
                          run / "fall_model_int8.tflite")
ergebnisse["tflite_kb"] = round(kb, 1)

fl.save_results(run, ergebnisse, config=fl.default_config(),
    notes=("Schwellwert und Random Forest auf rohen Fenstern, "
           "CNN auf z-normalisierten. Identischer Testsatz."))

Saved artifact at '/tmp/tmp2tpqsh8u'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 100, 6), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  136671331401168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136671028020432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136671028020240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136671028020816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136671028021200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136671028021008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136671028021776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136671028022736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136671028022544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136671028022352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136671028020624: Te

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


  int8-TFLite: 77.6 KB -> /content/drive/MyDrive/ML_Fall_Detection/results/01_basismodelle_2026-08-26_0838/fall_model_int8.tflite
[gesichert] /content/drive/MyDrive/ML_Fall_Detection/results/01_basismodelle_2026-08-26_0838/results.json


PosixPath('/content/drive/MyDrive/ML_Fall_Detection/results/01_basismodelle_2026-08-26_0838/results.json')